In [1]:
!nvidia-smi

Thu Apr 16 10:00:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip -q install transformers accelerate sentencepiece

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.10.0+cu128
True
Tesla T4


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)
model = model.to("cuda")
model.eval()

print("Loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded successfully


In [5]:
prompt = "Explain the role of attention in transformer models."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        use_cache=True
    )

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Explain the role of attention in transformer models. In a transformer model, attention is used to focus on relevant parts of the input data and improve the performance of the model. Attention mechanisms are typically applied at the


In [ ]:
%%writefile benchmark_minimal_skeleton.py
# Day 3 - Minimal LLM Benchmark Skeleton (Colab / Single T4)
# NOTE:
# - Core logic is intentionally left as TODOs.
# - Fill in the missing parts yourself.
# - Goal: run a small HF causal LM and record basic latency stats.

import time
import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any

import torch
# TODO: import the Hugging Face classes you need
# from transformers import ...


# =========================
# Config
# =========================

@dataclass
class BenchmarkConfig:
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"   # you may replace this
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    dtype: str = "float16"   # e.g. "float16", "bfloat16", "float32"
    max_new_tokens: int = 64
    num_runs: int = 3
    num_warmup: int = 1
    do_sample: bool = False
    temperature: float = 1.0
    top_p: float = 1.0
    use_kv_cache: bool = True
    results_dir: str = "./results"


# =========================
# Prompt helpers
# =========================

def build_base_prompt() -> str:
    """
    Return a base prompt string.
    Keep this simple for Day 3.
    """
    # TODO:
    # Return a short instruction-style prompt.
    # raise NotImplementedError
    return (
        "You are a helpful AI assistant. "
        "Please explain the role of attention in transformer models, "
        "including why attention is useful for long-range dependency modeling. "
    )



def make_prompt_near_target_tokens(
    tokenizer,
    base_text: str,
    target_prompt_len: int,
    max_iters: int = 50
) -> str:
    """
    Build a prompt whose tokenized length is close to target_prompt_len.

    Suggested strategy:
    - Repeatedly append base_text or filler text
    - Tokenize each time
    - Stop when token length reaches/exceeds target
    - Truncate by token if needed
    """
    # TODO:
    # 1. Construct a long enough prompt
    # 2. Tokenize it
    # 3. Trim it to target_prompt_len tokens
    # 4. Decode back to string if needed
    # raise NotImplementedError
    text = base_text
    filler = (
        "Add more explanation about self-attention, key-value interactions, "
        "and contextual representation learning. "
    )

    for _ in range(max_iters):
        ids = tokenizer(text, return_tensors="pt").input_ids[0]
        if ids.shape[0] >= target_prompt_len:
            ids = ids[:target_prompt_len]
            text = tokenizer.decode(ids, skip_special_tokens=True)
            return text
        text += filler
    
    ids = tokenizer(text, return_tensors="pt").input_ids[0][:target_prompt_len]
    return tokenizer.decode(ids, skip_special_tokens=True)



# =========================
# Model / tokenizer loading
# =========================

def resolve_torch_dtype(dtype_str: str):
    """
    Map config dtype string -> torch dtype.
    """
    mapping = {
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
        "float32": torch.float32,
    }
    if dtype_str not in mapping:
        raise ValueError(f"Unsupported dtype: {dtype_str}")
    return mapping[dtype_str]


def load_tokenizer_and_model(cfg: BenchmarkConfig):
    """
    Load tokenizer + causal LM and move model to device.
    """
    torch_dtype = resolve_torch_dtype(cfg.dtype)

    # TODO:
    # 1. Load tokenizer from cfg.model_name
    # 2. Load causal LM from cfg.model_name
    # 3. Move model to cfg.device if needed
    # 4. Set model.eval()
    # 5. Return tokenizer, model
    # raise NotImplementedError
    torch_type = resolve_torch_dtype(cfg.dtype)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        torch_dtype=torch_dtype,
    )

    model = model.to(cfg.device)
    model.eval()
    return tokenizer, model


# =========================
# Token counting helpers
# =========================

def count_input_tokens(tokenizer, prompt: str) -> int:
    """
    Return the number of input tokens for the prompt.
    """
    # TODO:
    # Tokenize prompt and return sequence length
    # raise NotImplementedError
    return tokenizer(prompt, return_tensors="pt").input_ids.shape[1]


def count_generated_tokens(input_ids, output_ids) -> int:
    """
    Given model input_ids and full output_ids, return newly generated token count.
    """
    # TODO:
    # Usually: generated_len = output_len - input_len
    raise NotImplementedError


# =========================
# Benchmark core
# =========================

def prepare_inputs(tokenizer, prompt: str, device: str):
    """
    Tokenize prompt and move tensors to device.
    """
    # TODO:
    # 1. Tokenize prompt with return_tensors='pt'
    # 2. Move tensors to device
    # 3. Return tokenized inputs
    raise NotImplementedError


def run_one_generation(
    model,
    tokenizer,
    prompt: str,
    cfg: BenchmarkConfig,
) -> Dict[str, Any]:
    """
    Run one generation and collect basic metrics.

    For Day 3, at minimum record:
    - input_tokens
    - output_tokens
    - total_latency_sec
    - output_tokens_per_sec
    - generated_text
    """
    inputs = prepare_inputs(tokenizer, prompt, cfg.device)

    # Optional but recommended for better timing accuracy on CUDA
    if cfg.device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        # TODO:
        # Call model.generate(...) with:
        # - max_new_tokens
        # - do_sample
        # - temperature / top_p if relevant
        # - use_cache
        # outputs = None
        if cfg.do_sample:
            outputs = model.generate(
                **inputs,
                max_new_tokens=cfg.max_new_tokens,
                do_sample=True,
                temperature=cfg.temperature,
                top_p=cfg.top_p,
                use_cache=cfg.use_kv_cache,
                pad_token_id=tokenizer.pad_token_id,
            )
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=cfg.max_new_tokens,
                do_sample=False,
                use_cache=cfg.use_kv_cache,
                pad_token_id=tokenizer.pad_token_id,
            )


    if cfg.device == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()
    total_latency_sec = end - start

    # TODO:
    # 1. Compute input token count
    # 2. Compute generated token count
    # 3. Decode generated text
    # 4. Compute tokens/sec
    # 5. Return a dict
    # raise NotImplementedError
    input_tokens = inputs["input_ids"].shape[1]
    generated_tokens = count_generated_tokens(inputs["input_ids"], outputs)

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_only_ids = outputs[0][input_tokens:]
    generated_text = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

    output_tokens_per_sec = (
        generated_tokens / total_latency_sec if total_latency_sec > 0 else 0.0
    )

    return {
        "input_tokens": int(input_tokens),
        "generated_tokens": int(generated_tokens),
        "total_latency_sec": float(total_latency_sec),
        "output_tokens_per_sec": float(output_tokens_per_sec),
        "generated_text": generated_text,
        "full_text": full_text,
    }


def run_warmup(
    model,
    tokenizer,
    prompt: str,
    cfg: BenchmarkConfig,
):
    """
    Warm up the model before measured runs.
    """
    for i in range(cfg.num_warmup):
        print(f"[Warmup {i+1}/{cfg.num_warmup}]")
        _ = run_one_generation(model, tokenizer, prompt, cfg)


def run_benchmark(
    model,
    tokenizer,
    prompt: str,
    cfg: BenchmarkConfig,
) -> List[Dict[str, Any]]:
    """
    Run measured benchmark iterations.
    """
    records = []

    for i in range(cfg.num_runs):
        print(f"[Run {i+1}/{cfg.num_runs}]")
        record = run_one_generation(model, tokenizer, prompt, cfg)

        # Add metadata useful for later plotting / analysis
        record["run_idx"] = i
        record["model_name"] = cfg.model_name
        record["device"] = cfg.device
        record["max_new_tokens"] = cfg.max_new_tokens
        record["do_sample"] = cfg.do_sample
        record["use_kv_cache"] = cfg.use_kv_cache

        records.append(record)

        print(json.dumps({
            "run_idx": record["run_idx"],
            "input_tokens": record.get("input_tokens"),
            "generated_tokens": record.get("generated_tokens"),
            "total_latency_sec": round(record.get("total_latency_sec", -1), 4),
            "output_tokens_per_sec": round(record.get("output_tokens_per_sec", -1), 4),
        }, indent=2))

    return records


# =========================
# Result saving
# =========================

def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)


def save_results_json(records: List[Dict[str, Any]], out_path: str):
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)


def summarize_records(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Compute simple averages over measured runs.
    """
    # TODO:
    # Average at least:
    # - total_latency_sec
    # - output_tokens_per_sec
    # - generated_tokens
    # raise NotImplementedError
    n = len(records)
    if n == 0:
        return {"num_runs" : 0}
    
    avg_total_latency_sec = sum(r["total_latency_sec"] for r in records) / n
    avg_output_tokens_per_sec = sum(r["output_tokens_per_sec"] for r in records) / n
    avg_generated_tokens = sum(r["generated_tokens"] for r in records) / n
    avg_input_tokens = sum(r["input_tokens"] for r in records) / n

    return {
        "num_runs": n,
        "avg_input_tokens": avg_input_tokens,
        "avg_generated_tokens": avg_generated_tokens,
        "avg_total_latency_sec": avg_total_latency_sec,
        "avg_output_tokens_per_sec": avg_output_tokens_per_sec,
    }



# =========================
# Main
# =========================

def main():
    cfg = BenchmarkConfig()

    print("=== Benchmark Config ===")
    print(cfg)

    ensure_dir(cfg.results_dir)

    tokenizer, model = load_tokenizer_and_model(cfg)

    base_prompt = build_base_prompt()

    # For Day 3 you can either:
    # A) use base_prompt directly
    # B) build a prompt close to a target token length
    target_prompt_len = 128

    # TODO:
    # Choose one:
    # prompt = base_prompt
    # OR
    # prompt = make_prompt_near_target_tokens(tokenizer, base_prompt, target_prompt_len)
    # raise NotImplementedError
    prompt = make_prompt_near_target_tokens(tokenizer, base_prompt, target_prompt_len)

    print("\n=== Prompt Preview ===")
    print(prompt[:300] + ("..." if len(prompt) > 300 else ""))

    # TODO:
    # Print actual input token count for verification
    # raise NotImplementedError
    actual_input = count_input_tokens(tokenizer, prompt)

    print("\n=== Warmup ===")
    run_warmup(model, tokenizer, prompt, cfg)

    print("\n=== Measured Runs ===")
    records = run_benchmark(model, tokenizer, prompt, cfg)

    summary = summarize_records(records)

    records_path = str(Path(cfg.results_dir) / "benchmark_records.json")
    summary_path = str(Path(cfg.results_dir) / "benchmark_summary.json")

    save_results_json(records, records_path)
    save_results_json(summary, summary_path)

    print("\n=== Summary ===")
    print(json.dumps(summary, indent=2))

    print("\nSaved:")
    print(records_path)
    print(summary_path)


if __name__ == "__main__":
    main()

In [ ]:
!python benchmark_minimal_skeleton.py